# Cumberland Mesh Optimization Workflow

This notebook uses the same style of mesh inputs that were used in the Cumberland project before the workflow moved over to AlgoMesh 2.

It does three things:

- builds a representative `TriangleGrid -> VoronoiGridPlus` mesh from the Cumberland domain, creeks, lakes, and infiltration facilities
- compares a baseline Triangle build with a safer cleanup-only rebuild
- optionally imports the AlgoMesh-exported `.disu` grid and compares its size with the Triangle/Voronoi versions

This is not a tiny demo. It is meant to feel like a real project-scale mesh workflow.

Important note: the `slopes.gpkg` refinement region is currently left out of the default build because it is the piece most likely to trigger a native `triangle.exe` access-violation crash on this Cumberland geometry. The notebook keeps that layer available as an explicit opt-in experiment instead of making it part of the default path.

A second note: the current `build_mesh(profile="balanced", ...)` CVT remeshing path is experimental and is **not** part of the default comparison because it can introduce quality regressions on large real meshes like Cumberland.

A third note: one old Cumberland refinement trick was a manual point region near `(1363943, 112693)`. In the current Triangle workflow that point region makes the whole grid dramatically denser, so it is also off by default in this notebook.


In [ ]:
from pathlib import Path
import time

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import simple_modflow as mf

cumberland_root = Path(r"C:/Users/lukem/mf6/Cumberland general")
boundary_root = cumberland_root / "Boundaries"
herrera_root = cumberland_root / "Herrera"
algomesh_root = cumberland_root / "algomesh"
surface_root = cumberland_root / "Surfaces"

domain_path = cumberland_root / "domain_v3.gpkg"
slopes_path = boundary_root / "slopes.gpkg"
deep_lake_path = boundary_root / "deep lake.gpkg"
hyde_lake_path = boundary_root / "hyde lake.gpkg"
deep_creek_path = boundary_root / "deep creek.gpkg"
hyde_creek_path = boundary_root / "hyde creek.gpkg"
sw_facilities_path = herrera_root / "Infiltration_Facilities.gpkg"
disu_path = algomesh_root / "cumb_vor_algomesh_facilities_rev2.1.disu"
top_raster = surface_root / "top_of_model_with_pits_no_overlap_with_botm_v5.tif"
botm_raster = surface_root / "cumb_aq_btm_v14c.tif"

required_paths = [
    domain_path,
    deep_lake_path,
    hyde_lake_path,
    deep_creek_path,
    hyde_creek_path,
    sw_facilities_path,
]
missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing Cumberland mesh inputs:\n" + "\n".join(str(path) for path in missing))

workspace = Path.cwd().resolve().parents[1] / "artifacts" / "cumberland_mesh_optimization"
workspace.mkdir(parents=True, exist_ok=True)

background_max_area = 40000
deep_creek_max_area = 200
hyde_creek_max_area = 200
lake_max_area = 2000
facility_max_area = 500

include_slopes_refinement = False
include_legacy_point_region = False
legacy_point_region_max_area = 2000
include_algomesh_comparison = disu_path.exists()
include_experimental_optimization = False
workspace


In [ ]:
def build_cumberland_triangle(mesh_name: str, *, mode: str = "baseline", include_slopes: bool = False):
    tri = mf.TriangleGrid(model_ws=str(workspace / mesh_name), angle=30)
    
    tri.set_domain_file(
        domain_path,
        buffer=-1,
        simplify_tolerance=10,
        densify_dist=25,
        max_area=background_max_area,
        label="domain",
    )
    tri.add_line_feature(
        deep_creek_path,
        buffer=20,
        simplify_tolerance=10,
        densify_dist=100,
        max_area=deep_creek_max_area,
        label="deep_creek",
        priority=4,
    )
    tri.add_line_feature(
        hyde_creek_path,
        buffer=20,
        simplify_tolerance=10,
        densify_dist=100,
        max_area=hyde_creek_max_area,
        label="hyde_creek",
        priority=4,
    )
    tri.add_region_file(
        deep_lake_path,
        simplify_tolerance=20,
        max_area=lake_max_area,
        label="deep_lake",
        priority=5,
    )
    tri.add_region_file(
        hyde_lake_path,
        simplify_tolerance=20,
        max_area=lake_max_area,
        label="hyde_lake",
        priority=5,
    )

    sw_facilities = mf.read_shp_gpkg(sw_facilities_path)
    for idx, poly in enumerate(sw_facilities.geometry):
        tri.add_region_polygon(
            poly,
            densify_dist=50,
            max_area=facility_max_area,
            buffer=-5,
            label=f"facility_{idx}",
            priority=6,
            source="facility",
        )

    if include_slopes:
        tri.add_region_file(
            slopes_path,
            max_area=4000,
            densify_dist=200,
            buffer=100,
            label="slopes",
            priority=2,
        )
    if include_legacy_point_region:
        tri.add_region(point=(1363943, 112693), maximum_area=legacy_point_region_max_area)

    start = time.perf_counter()
    if mode == "experimental_optimize":
        report = tri.build_mesh(
            profile="fast",
            protect_sources=("line",),
            protected_labels=["deep_creek", "hyde_creek", "deep_lake", "hyde_lake"],
            simplify_tolerance=2,
            target_segment_length=150,
            optimization_iterations=1,
            verbose=True,
        )
        quality = report["quality"]
    elif mode == "cleanup_only":
        tri.clean_geometry(
            simplify_tolerance=2,
            target_segment_length=150,
            resample_region_sources=("line",),
        )
        tri.build(verbose=False)
        quality = tri.quality_report(include_voronoi=True)
        report = {"cleanup": tri._last_cleanup_report, "optimization": None, "quality": quality}
    else:
        tri.build(verbose=False)
        quality = tri.quality_report(include_voronoi=True)
        report = {"cleanup": None, "optimization": None, "quality": quality}
    elapsed = time.perf_counter() - start
    report["elapsed_seconds"] = elapsed
    return tri, report


In [ ]:
tri_base, base_report = build_cumberland_triangle("triangle_baseline", mode="baseline", include_slopes=include_slopes_refinement)
tri_clean, clean_report = build_cumberland_triangle("triangle_cleanup_only", mode="cleanup_only", include_slopes=include_slopes_refinement)

comparison = pd.DataFrame(
    {
        "baseline": pd.Series(base_report["quality"]),
        "cleanup_only": pd.Series(clean_report["quality"]),
    }
)
comparison.loc[[
    "num_vertices",
    "num_triangles",
    "duplicate_vertex_count",
    "zero_area_triangle_count",
    "tiny_triangle_count",
    "sliver_triangle_count",
    "triangle_angle_min_overall",
    "triangle_edge_ratio_mean",
    "neighbor_area_ratio_mean",
    "voronoi_status",
    "voronoi_cell_count",
]]


In [ ]:
base_report["elapsed_seconds"], clean_report["elapsed_seconds"]


In [ ]:
vor_base = mf.VoronoiGridPlus(tri_base, rasters=[top_raster, botm_raster], crs="EPSG:2926", name="cumberland_base")
vor_clean = mf.VoronoiGridPlus(tri_clean, rasters=[top_raster, botm_raster], crs="EPSG:2926", name="cumberland_cleanup_only")

fig, axes = plt.subplots(1, 2, figsize=(16, 8), constrained_layout=True)
vor_base.gdf_vorPolys.boundary.plot(ax=axes[0], linewidth=0.15, color="black")
axes[0].set_title(f"Baseline Voronoi\nCells: {vor_base.ncpl:,}")
axes[0].set_aspect("equal")

vor_clean.gdf_vorPolys.boundary.plot(ax=axes[1], linewidth=0.15, color="black")
axes[1].set_title(f"Cleanup-only Voronoi\nCells: {vor_clean.ncpl:,}")
axes[1].set_aspect("equal")
plt.show()


In [ ]:
algomesh_summary = None
if include_algomesh_comparison:
    vor_algomesh = mf.VoronoiGridPlus.vor_from_disu(
        disu_path=disu_path,
        rasters=[top_raster, botm_raster],
        crs="EPSG:2926",
        name="cumberland_algomesh",
    )
    algomesh_summary = pd.DataFrame(
        {
            "grid": ["baseline_triangle", "cleanup_only_triangle", "algomesh_disu"],
            "cell_count": [vor_base.ncpl, vor_clean.ncpl, vor_algomesh.ncpl],
            "mean_cell_area": [
                float(np.mean(vor_base.get_cell_areas())),
                float(np.mean(vor_clean.get_cell_areas())),
                float(np.mean(vor_algomesh.get_cell_areas())),
            ],
        }
    )

algomesh_summary


In [ ]:
experimental_report = None
if include_experimental_optimization:
    tri_experimental, experimental_report = build_cumberland_triangle(
        "triangle_experimental_opt",
        mode="experimental_optimize",
        include_slopes=include_slopes_refinement,
    )
    pd.Series(experimental_report["quality"])[[
        "num_vertices",
        "num_triangles",
        "tiny_triangle_count",
        "sliver_triangle_count",
        "triangle_angle_min_overall",
        "triangle_edge_ratio_mean",
        "neighbor_area_ratio_mean",
        "voronoi_status",
    ]]
else:
    print("Experimental optimization is disabled by default because it can degrade large real meshes.")


In [ ]:
# From here, the optimized Voronoi grid can go directly into a model workflow.
# Example:
#
# model = mf.SimulationBase(name="cumberland_trial", mf_folder_path=workspace / "cumberland_trial", vor=vor_opt, nper=1)
#
# or save it for later:
# mf.pickle_dump(vor_opt, workspace / "cumberland_trial.vor")
#
# The main point of this notebook is to give you a realistic place to inspect:
# - whether the optimized mesh still respects the creek, lake, and facility features
# - whether cell count drops or rises materially
# - whether triangle/voronoi quality metrics improve before you trust it in MF6
# - whether it stays stable enough that you would actually prefer it to the imported AlgoMesh DISU grid
pass
